# Monitoring model behavior after deployment

_Doing ML for Real — the skills that matter_

**A model is not done at deploy — it quietly rots, and only monitoring tells you.**

A deployed model assumes tomorrow looks like the data it trained on. That assumption decays.

       Monitoring is the practice of measuring how far today's reality has drifted from training, and acting before the drift quietly wrecks your predictions.

---

This notebook is a practice scaffold for the **Monitoring model behavior after deployment** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q evidently river nannyml
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — evidently + river + nannyml

In [ ]:
# pip install evidently river nannyml
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 1) EVIDENTLY — batch input/prediction drift report (PSI + KS).
# ---------------------------------------------------------------
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

reference = pd.read_parquet("training_window.parquet")   # what the model trained on
current   = pd.read_parquet("last_24h.parquet")          # live traffic, recent window

report = Report(metrics=[DataDriftPreset()])             # PSI / KS / Wasserstein per column
report.run(reference_data=reference, current_data=current)
report.save_html("drift_report.html")
drift = report.as_dict()
print("dataset drift detected:", drift["metrics"][0]["result"]["dataset_drift"])

# ---------------------------------------------------------------
# 2) RIVER — online ADWIN concept-drift detector on the error stream.
# ---------------------------------------------------------------
from river import drift

adwin = drift.ADWIN()                  # Adaptive Windowing: auto-resizing change detector
errors = (current["prediction"] != current["label"]).astype(int)  # 1 if model was wrong
for i, err in enumerate(errors):
    adwin.update(err)                  # feed the live 0/1 error stream, one point at a time
    if adwin.drift_detected:
        print(f"ADWIN flagged concept drift at row {i}")  # window shrank -> error rate moved

# ---------------------------------------------------------------
# 3) NANNYML — estimate performance BEFORE labels arrive (CBPE).
# ---------------------------------------------------------------
import nannyml as nml

estimator = nml.CBPE(                  # Confidence-Based Performance Estimation
    y_pred="prediction", y_pred_proba="score", y_true="label",
    problem_type="classification_binary", metrics=["roc_auc"],
    chunk_size=500,
)
estimator.fit(reference)               # learns the score->performance mapping on labeled ref
estimated = estimator.estimate(current)  # predicts today's AUC with NO current labels needed
print(estimated.to_df()[["estimated_roc_auc"]])

# ---------------------------------------------------------------
# 4) PSI from scratch — the formula the report computes for you.
# ---------------------------------------------------------------
def psi(ref, cur, bins=10):
    q = np.quantile(ref, np.linspace(0, 1, bins + 1))   # quantile edges from reference
    q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(ref, q)[0] / len(ref)              # expected share per bin
    a = np.histogram(cur, q)[0] / len(cur)              # actual share per bin
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))       # sum (a_i - e_i) * ln(a_i / e_i)

print("PSI:", psi(reference["amount"].values, current["amount"].values))
# rule of thumb: <0.1 stable, 0.1-0.2 moderate, >0.2 significant drift -> alert + retrain

## Visualize the data & results

_On real data with a shift injected into later time windows, does the PSI drift signal cross the alert threshold exactly when the shift starts?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()                 # 569 real tumor records, 30 features
X, y = data.data, data.target
feat = list(data.feature_names).index("mean radius")

rng = np.random.RandomState(7)
order = rng.permutation(len(X)); X, y = X[order], y[order]

# Reference window: first 300 rows (also the training set).
ref_X, ref_y = X[:300], y[:300]
clf = LogisticRegression(max_iter=10000).fit(ref_X, ref_y)
ref = ref_X[:, feat]                          # reference distribution of the monitored feature

pool_X, pool_y = X[300:], y[300:]             # held-out pool for the live stream
nW, sz = 6, 120
std = ref.std()

def psi(ref, cur, bins=5):
    q = np.quantile(ref, np.linspace(0, 1, bins + 1)); q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(ref, q)[0] / len(ref)
    a = np.histogram(cur, q)[0] / len(cur)
    e, a = np.clip(e, 1e-3, None), np.clip(a, 1e-3, None)
    return float(np.sum((a - e) * np.log(a / e)))

wrng = np.random.RandomState(123)
psis = []
for w in range(nW):
    idx = wrng.choice(len(pool_X), sz, replace=True)
    wX = pool_X[idx].copy()
    if w >= 3:                                # inject a growing shift into windows 4,5,6
        wX[:, feat] = wX[:, feat] + std * 0.45 * (w - 2)
    psis.append(round(psi(ref, wX[:, feat]), 3))

print("PSI per window:", psis)               # [0.134, 0.051, 0.115, 0.329, 1.399, 2.484]
# threshold line at 0.2; the alert fires at window 4 where the shift begins.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your fraud model's input drift (PSI) is flat and low, but its accuracy has quietly dropped over the last month. The inputs look identical to training. What kind of drift is this, and which monitor would have caught it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that input drift is low — the feature distributions still match training. — _PSI / KS only compare input distributions; if they are stable, input drift is ruled out._
- Recognize the input&rarr;label relationship changed: the same transaction pattern now means something different (fraudsters adapted). — _That is concept drift, not covariate (input) drift — the mapping $P(y\mid x)$ moved while $P(x)$ stayed put._
- Run an online detector on the error stream (ADWIN or DDM), not just a distribution comparison on inputs. — _Concept drift is invisible to input-only monitors; it surfaces as a rising error rate that ADWIN / DDM flag automatically._

**Answer:** This is concept drift: $P(x)$ is stable so PSI stays low, but $P(y\mid x)$ has shifted, so accuracy falls. An input-drift monitor can never see it. You need a detector on the error stream — ADWIN (Adaptive Windowing) or DDM (Drift Detection Method) — plus performance tracking once labels arrive (and NannyML to estimate performance before they do).

</details>

**Problem 2.** Reference bins $e=[0.5,0.3,0.2]$; current window $a=[0.3,0.3,0.4]$. Compute the PSI and decide whether to alert (threshold 0.2).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bin 1: $(0.3-0.5)\ln(0.3/0.5)=(-0.2)(\ln 0.6)=(-0.2)(-0.511)=0.102$. — _Both factors are negative &rarr; positive contribution; this bin lost a lot of mass._
- Bin 2: $(0.3-0.3)\ln(0.3/0.3)=0$. — _No change in this bin contributes nothing._
- Bin 3: $(0.4-0.2)\ln(0.4/0.2)=(0.2)(\ln 2)=(0.2)(0.693)=0.139$. — _This bin gained mass; both factors positive._
- Sum: $0.102+0+0.139=0.241$. — _PSI adds the per-bin contributions._

**Answer:** PSI $=0.241$. That is above the $0.2$ threshold &rarr; alert: significant input drift. Mass has clearly moved from bin 1 to bin 3, and the metric flags it.

</details>

**Problem 3.** You have a new candidate model you think is better. Labels for your task take three weeks to arrive. How do you roll it out safely and measure it, given you cannot wait three weeks per step?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Deploy the candidate in shadow mode first: it scores live traffic but its outputs are logged, not served. — _Shadow gives you a real side-by-side on production traffic with zero user risk and no waiting for labels._
- Compare the candidate's score distribution and a label-free performance estimate (NannyML CBPE) against production. — _You get an early read on quality before any ground truth exists._
- Promote to a small canary slice, watch input/prediction drift and the estimator on that slice, then ramp up gradually. — _Canary limits blast radius; if monitors stay healthy you increase traffic, otherwise roll back instantly._

**Answer:** Shadow, then canary. Shadow-deploy the candidate to score live traffic without serving it, compare score distributions and a label-free estimate (NannyML's CBPE / Confidence-Based Performance Estimation), then canary to a small slice with the same monitors and ramp up only if they stay green. You never wait three weeks to get a signal, and a bad model never touches most users.

</details>